    Calculation of Paid Spread per Tradeи https://app.clickup.com/t/869bpwb2b
    Нужно подготовить отчет по сумме уплаченного спреда по сделкам, с учетом нашего спреда (маркапа).
    На staging-сервере будет предоставлена таблица сделок (SQL). Для каждой транзакции фиксируются поля:
    marketAsk, marketBid — “чистые” котировки от провайдера на момент сделки
    marketSpread — спред провайдера
    markup — рассчитанный маркап на момент сделки
    Требование:
    Добавить в выборку/таблицу вычисляемое поле: spread_with_markup_usd — уплаченный спред с маркапом, выраженный в USD.
    Примечания / упрощение:
    Для конвертации в USD можно использовать текущий курс (на момент формирования отчета), без исторических курсов.
    Ожидаемый результат:
    в таблицу сделок добаить колонку  spread_with_markup_usd по  каждой сделке

In [1]:
# --- SETTINGS ---

# Скрипт работает в двух режимах: тестовом и реальном.
IS_TEST_MODE = True  # Смени на False для работы с реальными данными

# ====================== BOOTSTRAP ======================
import os
import sys
from pathlib import Path

file_dir = os.getcwd()                                                  # */[sub_project_dir]/ipynb_files/...
#sub_project_dir = Path(file_dir).parent
#project_dir = sub_project_dir.parent
parent_dir = Path(file_dir).parent.parent.parent   # Поднимемся на 3 уровня выше, чтобы попасть в корень проекта

libraries_path = str(parent_dir / "libraries_py")
if libraries_path not in sys.path: sys.path.insert(0, libraries_path)

# ====================== ОСНОВНОЙ КОД ======================
from project_config import ProjectConfig

paths = ProjectConfig(file_dir=file_dir)

print(f"✅ Режим тестирования: {'ВКЛЮЧЕН' if IS_TEST_MODE else 'ВЫКЛЮЧЕН'}")
print(f"✅ Путь к файлу: {paths.input_samples_data}")
"""# Пример использования
print(paths.input_log_data)
print(paths.output_log_data)
print(paths.project_dir)"""

Заведомо существующие директории:
    📁 [file_dir]; Путь к директории с ipynb/py файлами: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Calculation_of_Paid_Spread_per_Trade\ipynb_files
    📁 [sub_project_dir]; Путь к директории СубПроекта: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Calculation_of_Paid_Spread_per_Trade
    📁 [project_dir]; Путь к директории Проекта: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform
    📁 [parent_dir]; Путь к директории для доступа к библиотекам: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations

Директории СубПроекта:
  Директории Исходных Данных:
❗📁 [input_log_data] создан: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Calculation_of_Paid_Spread_per_Trade\input_data\input_log_data
❗📁 [input_temp_data] создан: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Calculation_of_Paid_Spread_per_Trade\input_data\inp

'# Пример использования\nprint(paths.input_log_data)\nprint(paths.output_log_data)\nprint(paths.project_dir)'

In [2]:
# Динамический импорт и инициализация библиотек и путей для работы с данными в Python <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
from pathlib import Path
import pandas as pd
import sys
import os
import numpy as np
from pathlib import Path
import io

# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                                    # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)
if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                               # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else:
    print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                                           # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "pd_set_option",                        # Вывод ДФ
                #"df_to_csv",                            # Сохранение ДФ в CSV 
                "CSVLoader",
                "save_data_log_work_file",
                #"detect_encoding",
                #"time_to_minutes",
                #"load_string_list",
                "list_print",
                "move_column",
                #"time_str_to_unix_time_2",
                "list_to_str",
                #'save_dict_log_work_file',
                'save_int_list_data_log_work_file',
                #'select_non_numeric_rows',
                'merge_left_with_check'],           # Вызываем функцию слияния ДФ с проверкой наличия колонок и размерности
    "sql_request_2":
        [libraries_path, 
                "pd_read_sql"]
                }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import

print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 


 ✅ Импорт [dynamic_import_functions.py] успешен.
Импорт из 'yar_sed_general_lib' успешен: ['pd_set_option', 'CSVLoader', 'save_data_log_work_file', 'list_print', 'move_column', 'list_to_str', 'save_int_list_data_log_work_file', 'merge_left_with_check']
Импорт из 'sql_request_2' успешен: ['pd_read_sql']

 Импортированные функции и их параметры:
Функция 'pd_set_option' из модуля 'yar_sed_general_lib' ожидает параметры: name_df: str, df: pandas.core.frame.DataFrame, rows: int = 10, columns: int | None = None, min_rows: int | None = None, width: int = 100
Функция 'CSVLoader' из модуля 'yar_sed_general_lib' ожидает параметры: file_path, delimiter=';', encoding='utf-8', df_name='dataframe'
Функция 'save_data_log_work_file' из модуля 'yar_sed_general_lib' ожидает параметры: df, file_name, directory_data_temp_files, directory_data_log_files
Функция 'list_print' из модуля 'yar_sed_general_lib' ожидает параметры: lst, label=None, limit=10
Функция 'move_column' из модуля 'yar_sed_general_lib' ож

In [ ]:
# ====================== ЧТЕНИЕ SQL ФАЙЛА И ПРЕОБРАЗОВАНИЕ В DATAFRAME ======================

if IS_TEST_MODE:
    print(f"✅ Режим тестирования: ВКЛЮЧЕН. Используем тестовые данные из {paths.input_samples_data}")
    sql_file = Path(paths.input_samples_data, "transactions.sql")    # Путь к файлу
else:
    print(f"✅ Режим тестирования: ВЫКЛЮЧЕН. Используем реальные данные из {paths.input_temp_data}")
    sql_file = Path(paths.input_temp_data, "transactions.sql")        # Путь к файлу

sql_text = sql_file.read_text(encoding="utf-8")                                                                                 # 1. Читаем весь текст
values_text = sql_text.split("VALUES", 1)[1].strip().rstrip(";")                    # 2. Оставляем только VALUES (всё после первой скобки)
values_text = values_text.replace("),\n(", ")\n(").replace("),(", ")\n(")           # 3. Преобразуем в CSV-подобный формат; Заменяем "),(" на ")\n(" для разделения строк
lines = [line.strip("()") for line in values_text.splitlines()]                     # 4. Убираем внешние скобки каждой строки

df = pd.read_csv(io.StringIO("\n".join(lines)), header=None, quotechar="'", escapechar="\\")    # 5. Создаём DataFrame; Разделитель - запятая, но учтём кавычки
                 
columns = [                                                                                     # 6. Задаём имена колонок (взяли из INSERT)
    'id', 'timestamp', 'login', 'orderId', 'action', 'entry', 'reason', 'contractSize',
    'time', 'timeMsc', 'symbol', 'price', 'profit', 'swap', 'commission', 'margin',
    'positionId', 'comment', 'stopLoss', 'takeProfit', 'marketBid', 'marketAsk',
    'volume', 'volumeClosed', 'digits', 'marketSpread', 'markup', 'tickTimestampMsc',
    'spreadType', 'spreadAlgorithm'
]

df.columns = columns

df['symbol'] = df['symbol'].str.replace(r'^\s*["\']|["\']\s*$', '', regex=True)
numeric_cols = [                                                                        # Cписок колонок с числами в строковом формате
    'markup', 'marketSpread', 'action', 'entry', 'price', 'profit', 'swap', 'marketBid', 'marketAsk', 'volume', 'volumeClosed', 'contractSize']
for col in numeric_cols: df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')  # Преобразуем строки в float, убираем лишние пробелы

symbols = set(df['entry'])
print(symbols)

df = df[df['entry'] == 1].copy()                # Фильтруем только сделки ЗАКРЫТИЯ позиций/ т.к. нам не принципиально какой спред брать
df['spread_calc'] = np.where(df['action'] > 0,  # 
                                        (df['marketAsk'] + (df['price'] - df['marketAsk'])) - (df['marketBid'] - (df['price'] - df['marketAsk'])),
                                        (df['marketAsk'] + (df['marketBid'] - df['price'])) - (df['marketBid'] - (df['marketBid'] - df['price']))
                                    )
df['marketSpread'] = df['marketAsk'] - df['marketBid']
df['spread_check'] = df['spread_calc'] / df['marketSpread']
df = df.sort_values(by='spread_check', ascending=False)

# Перемещение колонок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<                                        
new_columns_list = ['symbol', 'spread_check', 'spread_calc', 'marketSpread', 'price', 'marketBid', 'marketAsk'] # список колонок
df = imported['move_column'](df, new_columns_list,  new_position_step=1, add_list_col_name = False) # Перемещаем колонки / Добавляем новые колонки,

df_filter = df[abs(df['spread_check']) > 10].copy()
symbol_set = set(df_filter['symbol'])
imported['list_print'](list(symbol_set),  "Символы с аномальным спредом")
imported["pd_set_option"] ("[ df_filter ]",  df_filter, 5)

list_traders_df = df.copy()
imported["pd_set_option"] ("[ list_traders_df ]",  list_traders_df, 5)


✅ Режим тестирования: ВКЛЮЧЕН. Используем тестовые данные из c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Calculation_of_Paid_Spread_per_Trade\input_data\input_samples


{0.0, 1.0, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan}
 
 [dif] Функция добавления новых колонок с пустыми значениями и изменения положения этих колонок  
 list_col_name = ['symbol', 'spread_check', 'spread_calc', 'marketSpread', 'price', 'marketBid', 'marketAsk'];  
 new_position = 0, new_position_step = 1
📝 [ 31 ] элементов в списке [ Символы с аномальным спредом ] список: ['USDJPY', 'SP500', 'SSPG.L', 'EURCHF', 'COPN.S', 'USDCAD', 'BRTSPOT', 'XAUUSD', 'USDRUB', 'AUDTRL']...

[ df_filter ]  (285 строк × 32 колонок)


,symbol,spread_check,spread_calc,marketSpread,price,marketBid,marketAsk,id,timestamp,login,orderId,action,entry,reason,contractSize,time,timeMsc,profit,swap,commission,margin,positionId,comment,stopLoss,takeProfit,volume,volumeClosed,digits,markup,tickTimestampMsc,spreadType,spreadAlgorithm
1292,FCHI,1098.288889,1482.69,1.35,7375.86,8116.53,8117.88,1074189,1765903751951,266972,0,0.0,1.0,1,100.0,'2025-12-16 16:49:12',1765903751951,-5641.80,-550.0,0.00000000,163.71000000,396657,'',0.00000000,0.00000000,0.07,0.07,2,0.675,1765903751837,'dynamic','percent'
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14951,FCHI,-1398.000000,-1537.80,1.10,7346.75,8115.10,8116.20,1082188,1767181689158,307478,0,1.0,1.0,1,100.0,'2025-12-31 11:48:09',1767181689158,-60.32,0.0,0.00000000,86.49000000,426601,'',0.00000000,0.00000000,0.04,0.04,2,0.550,1767181688968,'dynamic','percent'



[ list_traders_df ]  (7,842 строк × 32 колонок)


,symbol,spread_check,spread_calc,marketSpread,price,marketBid,marketAsk,id,timestamp,login,orderId,action,entry,reason,contractSize,time,timeMsc,profit,swap,commission,margin,positionId,comment,stopLoss,takeProfit,volume,volumeClosed,digits,markup,tickTimestampMsc,spreadType,spreadAlgorithm
1292,FCHI,1098.288889,1482.69,1.35,7375.86,8116.53,8117.88,1074189,1765903751951,266972,0,0.0,1.0,1,100.0,'2025-12-16 16:49:12',1765903751951,-5641.80,-550.0,0.00000000,163.71000000,396657,'',0.00000000,0.00000000,0.07,0.07,2,0.675,1765903751837,'dynamic','percent'
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14951,FCHI,-1398.000000,-1537.80,1.10,7346.75,8115.10,8116.20,1082188,1767181689158,307478,0,1.0,1.0,1,100.0,'2025-12-31 11:48:09',1767181689158,-60.32,0.0,0.00000000,86.49000000,426601,'',0.00000000,0.00000000,0.04,0.04,2,0.550,1767181688968,'dynamic','percent'


In [13]:
# Загрузка файла с торговыми условиями по торговым инструментам <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# Данные формируются в файле trading_conditions.ipynb в блоке # Фильтруем ДФ с торговыми условиями '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
df = list_traders_df.copy()

files_name = "trading_settings_sales_basic_df.csv"
file_path_symbols = Path(paths.input_samples_data, files_name)   # Файл c торговыми условиями
print("trading_settings_sales_basic_df.csv:", file_path_symbols)
loader = imported["CSVLoader"](file_path_symbols, delimiter=',', encoding='utf-8', df_name='my_dataframe')  # CSVLoader для загрузки данных из файла
trading_settings_sales_basic_df = loader.load_data()
#imported["pd_set_option"] (f"Торговые условия из, загруженного выше, файла {files_name} [ trading_settings_sales_basic_df ]",  trading_settings_sales_basic_df, 5)

cols_map = {"client_spread_money_lot": "spread_money_lot", "point_money":"point_money_", "precision_s":"precision_s_", "contractSize": "contractSize_"}
df, not_found, not_used = imported['merge_left_with_check']( # Вызываем функцию слияния ДФ с проверкой наличия колонок и размерности
    df_left= df, df_right= trading_settings_sales_basic_df, left_on= "symbol", right_on= "name_s",  cols_map=cols_map)

df['pos_price'] = df['volume'] * df['spread_money_lot']

df['spread_calc_point'] = df['spread_calc'] * (10** df['precision_s_'])

df['pos_price_2'] = pd.to_numeric(df['spread_calc_point'] * df['point_money_']) * df['contractSize'] * df['volume']
df['pos_price_2'] = df['pos_price_2'].map('{:.5f}'.format)
df['pos_price_3'] = (df['markup'] + df['marketSpread']) * (10** df['precision_s_']) * df['volume'] * df['point_money_'] * df['contractSize_']

df['point_money_'] = df['point_money_'].map('{:.5f}'.format)

"""# Перемещение колонок <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<                                        
new_columns_list = ['spread_calc','spread_calc_point','point_money_', 'pos_price', 'pos_price_2'] # список колонок
df = imported['move_column'](df, new_columns_list,  new_position_step=1, add_list_col_name = False) # Перемещаем колонки / Добавляем новые колонки,"""


list_col_name = ['markup','marketSpread','pos_price_3', 'contractSize_', 'contractSize', 'volume', 'spread_calc','spread_calc_point', 'pos_price_2', 'point_money_', 'spread_money_lot', 'pos_price']
df = imported["move_column"](df, list_col_name).copy()

df_filter = df[df['name_s'] == 'EURUSD'].copy()
imported["pd_set_option"] ("[ df_filter ]",  df_filter, 10)

imported["save_data_log_work_file"](df, "report_df.csv", paths.output_temp_data, paths.output_log_data)
print("Сумма уплаченного спреда", df['pos_price_3'].sum())
imported["pd_set_option"] ("[ df ]",  df, 100)

trading_settings_sales_basic_df.csv: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Calculation_of_Paid_Spread_per_Trade\input_data\input_samples\trading_settings_sales_basic_df.csv
✅ Success: [class CSVLoader]: DataFrame 'my_dataframe' успешно создан из 'c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Calculation_of_Paid_Spread_per_Trade\input_data\input_samples\trading_settings_sales_basic_df.csv'.

 [dif] Функция слияния ДФ с проверкой наличия колонок и размерности: merge_left_with_check()
 
 [dif] Функция добавления новых колонок с пустыми значениями и изменения положения этих колонок  
 list_col_name = ['markup', 'marketSpread', 'pos_price_3', 'contractSize_', 'contractSize', 'volume', 'spread_calc', 'spread_calc_point', 'pos_price_2', 'point_money_', 'spread_money_lot', 'pos_price'];  
 new_position = 0, new_position_step = 1

[ df_filter ]  (367 строк × 41 колонок)


,markup,marketSpread,pos_price_3,contractSize_,contractSize,volume,spread_calc,spread_calc_point,pos_price_2,point_money_,spread_money_lot,pos_price,symbol,spread_check,price,marketBid,marketAsk,id,timestamp,login,orderId,action,entry,reason,time,timeMsc,profit,swap,commission,margin,positionId,comment,stopLoss,takeProfit,volumeClosed,digits,tickTimestampMsc,spreadType,spreadAlgorithm,name_s,precision_s_
317,0.00005,0.00010,61.50,100000.0,100000.0,4.10,0.00042,42.0,172.20000,0.00001,54.6,223.860,EURUSD,4.200000,1.17440,1.17456,1.17466,1071153,1765463042593,261961,0,0.0,1.0,3,'2025-12-11 14:24:03',1765463042593,1459.60,0.00,0.00000000,1199.96000000,395450,'',0.00000000,1.17440000,4.10,5,1765463042221,'dynamic','percent',EURUSD,5.0
355,0.00005,0.00012,1.19,100000.0,100000.0,0.07,0.00044,44.0,3.08000,0.00001,54.6,3.822,EURUSD,3.666667,1.16608,1.16624,1.16636,1070171,1765393210528,308576,0,0.0,1.0,3,'2025-12-10 19:00:11',1765393210528,7.14,0.00,0.00000000,20.39000000,421980,'',0.00000000,1.16608000,0.07,5,1765393210260,'dynamic','percent',EURUSD,5.0
401,0.00005,0.00010,7.50,100000.0,100000.0,0.50,0.00034,34.0,17.00000,0.00001,54.6,27.300,EURUSD,3.400000,1.17500,1.17512,1.17522,1071341,1765467735464,302632,0,0.0,1.0,3,'2025-12-11 15:42:15',1765467735464,684.00,0.00,0.00000000,145.15000000,402996,'',0.00000000,1.17500000,0.50,5,1765467735009,'dynamic','percent',EURUSD,5.0
402,0.00005,0.00010,0.15,100000.0,100000.0,0.01,0.00034,34.0,0.34000,0.00001,54.6,0.546,EURUSD,3.400000,1.17500,1.17512,1.17522,1071340,1765467735464,653343,0,0.0,1.0,3,'2025-12-11 15:42:15',1765467735464,13.51,-1.60,0.00000000,2.88000000,411965,'',0.00000000,1.17500000,0.01,5,1765467735009,'dynamic','percent',EURUSD,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7579,0.00005,0.00013,0.36,100000.0,100000.0,0.02,0.00019,19.0,0.38000,0.00001,54.6,1.092,EURUSD,1.461538,1.18000,1.18003,1.18016,1079390,1766567829652,270330,0,0.0,1.0,3,'2025-12-24 09:17:10',1766567829652,1.58,0.00,0.00000000,5.90000000,426246,'',0.00000000,1.18000000,0.02,5,1766567828978,'dynamic','percent',EURUSD,5.0
7580,0.00005,0.00013,0.18,100000.0,100000.0,0.01,0.00019,19.0,0.19000,0.00001,54.6,0.546,EURUSD,1.461538,1.18000,1.18003,1.18016,1078564,1766492886847,297371,0,0.0,1.0,3,'2025-12-23 12:28:07',1766492886847,1.62,0.00,0.00000000,2.95000000,425778,'',0.00000000,1.18000000,0.01,5,1766492886006,'dynamic','percent',EURUSD,5.0
7589,0.00002,0.00010,6.00,100000.0,100000.0,0.50,0.00014,14.0,7.00000,0.00001,54.6,27.300,EURUSD,1.400000,1.17450,1.17438,1.17448,1082025,1767128123048,662822,0,1.0,1.0,3,'2025-12-30 20:55:23',1767128123048,89.50,-30.98,0.00000000,147.26000000,425634,'',0.00000000,1.17450000,0.50,5,1767128122768,'dynamic','percent');,EURUSD,5.0
7594,0.00002,0.00010,4.80,100000.0,100000.0,0.40,0.00014,14.0,5.60000,0.00001,54.6,21.840,EURUSD,1.400000,1.17869,1.17871,1.17881,1080263,1766763294297,662822,0,0.0,1.0,1,'2025-12-26 15:34:54',1766763294297,18.00,-2.00,0.00000000,0.00000000,426423,'',0.00000000,1.18000000,0.40,5,1766763294097,'dynamic','percent',EURUSD,5.0


💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Calculation_of_Paid_Spread_per_Trade\output_data\output_temp_data\report_df.csv
💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\Calculation_of_Paid_Spread_per_Trade\output_data\output_log_data\2026-07-30 10-17-15.report_df.csv
Сумма уплаченного спреда 369787.11940160627

[ df ]  (7,842 строк × 41 колонок)


,markup,marketSpread,pos_price_3,contractSize_,contractSize,volume,spread_calc,spread_calc_point,pos_price_2,point_money_,spread_money_lot,pos_price,symbol,spread_check,price,marketBid,marketAsk,id,timestamp,login,orderId,action,entry,reason,time,timeMsc,profit,swap,commission,margin,positionId,comment,stopLoss,takeProfit,volumeClosed,digits,tickTimestampMsc,spreadType,spreadAlgorithm,name_s,precision_s_
0,0.67500,1.3500,16.431447,100.0,100.0,0.07,1482.69000,148269.0,12030.98405,0.01159,320.224856,22.415740,FCHI,1098.288889,7375.86000,8116.53000,8117.88000,1074189,1765903751951,266972,0,0.0,1.0,1,'2025-12-16 16:49:12',1765903751951,-5641.80,-550.00,0.00000000,163.71000000,396657,'',0.00000000,0.00000000,0.07,2,1765903751837,'dynamic','percent',FCHI,2.0
1,0.67500,1.3500,4.694699,100.0,100.0,0.02,1482.69000,148269.0,3437.42402,0.01159,320.224856,6.404497,FCHI,1098.288889,7379.61000,8120.28000,8121.63000,1079296,1766565100095,206039,0,0.0,1.0,5,'2025-12-24 08:31:40',1766565100095,-666.82,-750.00,0.00000000,44.53000000,421214,'',0.00000000,8300.00000000,0.02,2,1766565099352,'dynamic','percent',FCHI,2.0
2,0.67500,1.3500,4.694699,100.0,100.0,0.02,1482.69000,148269.0,3437.42402,0.01159,320.224856,6.404497,FCHI,1098.288889,7379.61000,8120.28000,8121.63000,1079300,1766565100130,206039,0,0.0,1.0,5,'2025-12-24 08:31:40',1766565100130,-538.10,-400.00,0.00000000,44.24000000,421563,'',0.00000000,8350.00000000,0.02,2,1766565099352,'dynamic','percent',FCHI,2.0
3,0.75000,1.5000,20.865330,100.0,100.0,0.08,1543.00000,154300.0,14308.97964,0.01159,320.224856,25.617989,FCHI,1028.666667,7344.15000,8114.90000,8116.40000,1082187,1767181682437,307478,0,0.0,1.0,1,'2025-12-31 11:48:02',1767181682437,-3018.87,-7000.00,0.00000000,178.29000000,421109,'',0.00000000,8300.00000000,0.08,2,1767181681000,'dynamic','percent',FCHI,2.0
4,0.67500,1.3500,23.473496,100.0,0.0,0.10,1282.69000,128269.0,0.00000,0.01159,320.224856,32.022486,FCHI,950.140741,7416.66000,8057.33000,8058.68000,1070724,1765446106783,187498,0,0.0,1.0,5,'2025-12-11 09:41:47',1765446106783,1972.84,-2601.00,0.00000000,202.85000000,350612,'',0.00000000,0.00000000,0.10,2,1765446105670,'dynamic','percent',FCHI,2.0
5,0.67500,1.3500,58.683741,100.0,100.0,0.25,882.69000,88269.0,25580.02519,0.01159,320.224856,80.056214,FCHI,653.844444,7669.26000,8109.93000,8111.28000,1068474,1765217060242,265357,0,0.0,1.0,1,'2025-12-08 18:04:20',1765217060242,292.41,0.00,0.00000000,556.44000000,421144,'',0.00000000,0.00000000,0.25,2,0,'dynamic','percent',FCHI,2.0
6,0.67500,1.3500,4.694699,100.0,0.0,0.02,882.69000,88269.0,0.00000,0.01159,320.224856,6.404497,FCHI,653.844444,7682.21000,8122.88000,8124.23000,1067151,1764945407610,269204,0,0.0,1.0,5,'2025-12-05 14:36:48',1764945407610,3516.24,-450.00,0.00000000,36.10000000,352157,'',0.00000000,0.00000000,0.02,2,1764945406344,'dynamic','percent',FCHI,2.0
7,0.67500,1.3500,140.840982,100.0,100.0,0.60,880.01000,88001.0,61205.66555,0.01159,320.224856,192.134920,FCHI,651.859259,7689.00000,8128.33000,8129.68000,1068681,1765268504417,269337,0,0.0,1.0,3,'2025-12-09 08:21:44',1765268504417,2045.51,0.00,0.00000000,1336.71000000,421074,'',0.00000000,7689.00000000,0.60,2,1765268504321,'dynamic','percent',FCHI,2.0
8,0.55000,1.1000,1.650000,100.0,100.0,0.01,602.20000,60220.0,602.20000,0.01000,141048.000000,1410.480000,DJI,547.454545,48842.10000,48540.45000,48541.55000,1081035,1767019777603,69806,0,1.0,1.0,1,'2025-12-29 14:49:38',1767019777603,158.80,0.00,0.00000000,490.01000000,426890,'',0.00000000,48.60000000,0.01,2,1767019777519,'dynamic','percent',DJI,2.0
9,0.55000,1.1000,1.650000,100.0,100.0,0.01,601.20000,60120.0,601.20000,0.01000,141048.000000,1410.480000,DJI,546.545455,48690.10000,48388.95000,48390.05000,1081745,1767105732123,69806,0,1.0,1.0,2,'2025-12-30 14:42:12',1767105732123,76.30,0.00,0.00000000,487.66000000,427220,'',48690.00000000,48250.00000000,0.01,2,1767105731706,'dynamic','percent',DJI,2.0
